# Concours MTH3302 A2024 
### Simon Bachand, Jérémie Bolduc, Jacqueline Koch, Julien Roux

## Prédiction de la consommation en carburant de voitures récentes.

### Contexte
Une gestion efficace de la consommation de carburant devient un enjeu crucial tant pour les conducteurs que pour l’industrie automobile, particulièrement dans le contexte actuel de transition énergétique et de réduction des émissions de gaz à effet de serre. La consommation en carburant des véhicules récents dépend de plusieurs caractéristiques techniques telles que la boîte de vitesses, la cylindrée, le nombre de cylindres et le type de transmission. Ces variables influencent directement l'efficacité énergétique et peuvent varier d’un véhicule à l’autre.

### Objectif

Dans cette étude, nous nous concentrons sur la prédiction de la consommation en carburant de voitures récentes. À partir d’un jeu de données comprenant la consommation moyenne en litres pour 100 kilomètres (L/100km) de près de 400 véhicules, ainsi que leurs caractéristiques techniques, l’objectif est de prédire la consommation en carburant pour un ensemble de test en fonction de ces différentes caractéristiques. Ce modèle prédictif permettra d’évaluer plus précisément les performances de consommation des véhicules et d'aider à identifier les facteurs déterminants pour l’optimisation de la consommation de carburant.

### Variables

La variable d'intérêt est la **consommation** en L/100km.

Les variables explicatives sont les suivantes:
- année: l'année du modèle
- type: le type de véhicule
- nombre_cylindres: le nombre de cylindres du moteur
- cylindree: la cylindrée du moteur en L
- transmission: le type de transmission (propulsion, traction, 4x4 et intégrale)
- boite: le type de boite de vitesses (automatique ou manuelle)



## Chargement des données et des librairies

Importation des librairies utilisées dans le calepin.

In [ ]:
using CSV
using GLM
using MLJ
using DataFrames
using Gadfly
using StatsPlots
using Random
using Statistics
using Combinatorics
using LinearAlgebra
using HypothesisTests
using StatsModels
using CategoricalArrays
using StatsBase

Charger les données d'entrainement et de test.

In [ ]:
train = CSV.read("train.csv", DataFrame, decimal=',')
test = CSV.read("test.csv", DataFrame, decimal=',')

first(train, 5)

## Exploration des données

Nous avons commencé par examiner la structure du jeu de données d'entraînement pour mieux comprendre les variables disponibles et leurs types. À cette étape, l'objectif était de nous familiariser avec les données afin de déterminer leur pertinence et leur capacité à nous aider dans notre analyse. 

In [ ]:
describe(train)

Les données incluent des variables qui peuvent être utilisés pour prédire la consommation de carburant de différentes manières.

**Variables numériques (comme l'année de fabrication, le nombre de cylindres, la cylindrée) :**

Ces variables peuvent être utilisées pour comprendre l'impact de la technologie et de la taille du moteur sur la consommation. Par exemple, un moteur plus puissant (avec un nombre de cylindres plus élevé ou une cylindrée plus grande) pourrait être lié à une consommation plus élevée.

**Variables catégorielles (comme le type de véhicule, la transmission, et la boîte de vitesses) :**

Ces variables peuvent influencer la consommation en fonction du type de conduite. Par exemple, un véhicule avec une transmission 4x4 pourrait consommer plus de carburant que celui avec une transmission à traction. De même, la présence d'une boîte automatique peut avoir un impact sur la consommation en fonction des conditions de conduite.

Dans la partie 1, nous avons analysé l'influence de chaque variable sur la consommation.

---

Nous avons aussi utilisé un histogramme pour mieux comprendre la distribution des valeurs de consommation de carburant dans l'ensemble de données. Cette visualisation permet d'identifier la tendance centrale, l'étalement et la répartition de la consommation de carburant.

In [ ]:
plot(train, x=:consommation, Geom.histogram(bincount=15), Guide.xlabel("Consommation en L/100km"), Guide.ylabel("Consommation"))

La forme de l'histogramme suggère un pic autour du milieu de l'échelle, ce qui indique que la plupart des voitures ont une consommation de carburant modérée. La hauteur des barres diminue au fur et à mesure que l'on se rapproche des extrêmes inférieurs et supérieurs, ce qui indique qu'il y a moins de voitures ayant une consommation de carburant très faible ou très élevée. On peut reconnaître la forme d'une cloche de Gauss. La distribution semble quelque peu symétrique, mais il y a une baisse notable de la fréquence au milieu, ce qui suggère la possibilité d'une lacune ou d'une irrégularité. Dans l'ensemble, la majorité des voitures se regroupent autour 10L/100km, avec quelques valeurs aberrantes.

In [ ]:
# Afficher des diagrammes en boîte de consommation pour les variables du jeu de données
function box_plot_categorical(data)
    Gadfly.set_default_plot_size(30cm, 35cm)
    p1 = plot(data, x=:annee, y=:consommation, Geom.boxplot, Guide.title("Consommation par Année"), Guide.xlabel("Année"), Guide.ylabel("Consommation"))
    p2 = plot(data, x=:type, y=:consommation, Geom.boxplot, Guide.title("Consommation par Type de voiture"), Guide.xlabel("Type de voiture"), Guide.ylabel("Consommation"))
    p3 = plot(data, x=:nombre_cylindres, y=:consommation, Geom.boxplot, Guide.title("Consommation par nombre de cylindre"), Guide.xlabel("Nombre de cylindre"), Guide.ylabel("Consommation"))
    p4 = plot(data, x=:transmission, y=:consommation, Geom.boxplot, Guide.title("Consommation par type de transmission"), Guide.xlabel("Type de transmission"), Guide.ylabel("Consommation"))
    p5 = plot(data, x=:cylindree, y=:consommation, Geom.point, Geom.smooth(method=:lm), Guide.title("Consommation par cylindree"), Guide.xlabel("Cylindree"), Guide.ylabel("Consommation"))
    p6 = plot(data, x=:boite, y=:consommation, Geom.boxplot, Guide.title("Consommation par type de boite de vitesse"), Guide.xlabel("Boite de vitesse"), Guide.ylabel("Consommation"))

    grid = vstack(hstack(p1, p2), hstack(p3, p4), hstack(p5, p6))
    display(grid)

    # réinitialiser la taille pour ne pas affecter les autres graphiques
    Gadfly.set_default_plot_size(20cm, 15cm)
end

In [ ]:
box_plot_categorical(train)

Ces différents graphiques illustrent la relation entre la consommation et chaque variable du jeu de données.   
Ils montrent que la consommation est influencée par les variables telles que l'année, le type de voiture, le nombre de cylindres, le type de transmission, la cylindrée et le type de boîte de vitesses.  
On peut noter la consommation suit une relation relativement linéaire avec la cylindrée et le nombre de cylindres ce qui peut indiquer qu'une régression linéaire pourrait être un bon modèle pour prédire la consommation.  

On note aussi la présence de **valeurs aberrantes** dans les données, représentées par des points isolés dans les graphiques.  
Ces valeurs aberrantes peuvent affecter la performance du modèle de régression linéaire. Il faudra les traiter avant de construire le modèle.   

---

### Préparation des données

In [ ]:
# TODO A voir ce que l'on utilise pour notre meilleur prediction 

# Partie 1
## Régressions linéaires simples

Pour analyser l'influence de chaque variable sur la consommation, nous avons fait des régressions linéaires simples.
Pour chacune des six variables nous avons vérifié les quatres hypotheses pour la régression linéaire:
- hypothèse de linéarite
- hypothèse de normalité des erreurs
- hypothèse d'homoscédasticité des erreurs
- hypothèse d'indépendance des erreurs.

Avec le valeur R^2 nous avons testé si les variables ont un pouvoir explicatif signicatif sur la consommation d'essence d'une voiture.

In [ ]:
# y = train.consommation
# n = length(y)

In [ ]:
function compute_residuals(model, y)
    ŷ = StatsModels.predict(model)
    res = (y - ŷ) / std(ŷ)

    return res
end

function plot_explanatory_variable(model, data, xlabel)
    predictions = StatsModels.predict(model)

    Gadfly.plot(
        x = data.x, 
        y = data.y,
        layer(
            x = data.x,
            y = predictions,
            Geom.line,
            Theme(default_color="red"),
        ),
        Geom.point,
        Guide.xlabel(xlabel), 
        Guide.ylabel("Consommation d'essence (L/100km)", orientation=:vertical),
    )
end

function shapiro_wilk_test(model, data)
    errors = compute_residuals(model, data.y)
    p = pvalue(ShapiroWilkTest(errors))

    if p > 0.05
        println("$p > 0.05 -> On accepte l'hypothèse que les données proviennent d'une distribution normale")
    else 
        println("$p ≤ 0.05 -> On rejette l'hypothèse que les données proviennent d'une distribution normale")
    end
end

function residuals_vs_fitted_values_plot_test(model, data)
    errors = compute_residuals(model, data.y)
    fitted_values = fitted(model)

    Gadfly.plot(
        layer(x=fitted_values, y=errors, Geom.point),
        Guide.xlabel("Valeurs prédites"),
        Guide.ylabel("Résidus"),
    )
end

function residuals_vs_observation_order_plot_test(model, data)
    errors = compute_residuals(model, data.y)

    Gadfly.plot(
        layer(x=1:length(errors), y=errors, Geom.point),
        Guide.xlabel("Index"),
        Guide.ylabel("Résidus"),
    )
end

function get_data_set(seed, features, preprocess::Function = data -> nothing)
    Random.seed!(seed)
    data = CSV.read("train.csv", DataFrame, decimal=',')

    preprocess(data)
    # Séparer les données en un ensemble d'entraînement (80%) et un ensemble de validation (20%)
    train_id = sample(1:nrow(data), round(Int, .8*nrow(data)), ordered=true, replace=false)
    valid_id = setdiff(1:nrow(data), train_id)

    train = data[train_id,:]
    train = remove_outliers(train, features, :consommation)
    valid = data[valid_id,:]

    return train, valid
end

function linear_model(data, features::Vector{Symbol}, target::Symbol=:y)
    formula = Term(target) ~ sum(Term(feature) for feature in filter(x -> x != target, features))

    model = lm(formula, data)

    return model
end

function align_categorical_features(train, valid, features)
    data = vcat(train, valid)

    categorical_features = filter(x -> eltype(data[:, x]) <: AbstractString || eltype(data[:, x]) <: CategoricalValue, features)
    for feature in categorical_features
        train_values = unique(train[!, feature])
        valid_values = unique(valid[!, feature])

        train_indices_to_remove = findall(x -> !(x in valid_values), train[!, feature])
        train = train[setdiff(1:nrow(train), train_indices_to_remove), :]

        valid_indices_to_remove = findall(x -> !(x in train_values), valid[!, feature])
        valid = valid[setdiff(1:nrow(valid), valid_indices_to_remove), :]
    end

    return train, valid
end

In [ ]:
include("Jeremie_Utils.jl")

## 1.1 Analyse de la variable _nombre_cylindres_

In [ ]:
x = float.(train.nombre_cylindres)
data = DataFrame(y = train.consommation, x = x)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

On observe une linéarité entre le nombre de cylindres du véhicule ainsi que sa consommation d'essence 

In [ ]:
plot_explanatory_variable(model, data, "Nombre de cylindres")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont pas distribués normalement 

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homoscédasticité des erreurs**

On peut voir que les erreurs ne sont pas constantes selon le nombre de cylindres. Cependant, cela pourrait être causé par la représentation de catégorie de nombre de cylindres disproportionnée.

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

À l'exception de quelques points aberrants, les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

Le nombre de cylindres a un pouvoir explicatif significatif sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.2 Analyse de la variable _type_

In [ ]:
data = DataFrame(y = train.consommation, x = train.type)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

Puisqu'il sagit d'une variable explicative catégorielle nominale, l'hypothèse de linéarité entre les catégories est automatiquement à rejeter

In [ ]:
plot_explanatory_variable(model, data, "Type")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homoscédasticité des erreurs**

On peut voir que les erreurs ne sont pas constantes selon le type de voiture.

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

À l'exception de quelques points aberrants, les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

Le type de la voiture a un pouvoir explicatif modéré sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.3 Analyse de la variable _cylindree_

In [ ]:
data = DataFrame(y = train.consommation, x = train.cylindree)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

On voit qu'il existe une relation linéaire entre la cylindrée du moteur et la consommation d'essence de la voiture

In [ ]:
plot_explanatory_variable(model, data, "Cylindrée")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homoscédasticité des erreurs**

La variance des erreurs semble relativement constante, mise à part quelques points aberrants et la sur représentation des moteurs avec un cylindrée de 2 litres. 

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

À l'exception de quelques points aberrants, les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

La cylindrée du moteur a un pouvoir explicatif significatif sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.4 Analyse de la variable _transmission_

In [ ]:
x = train_data.transmission
data = DataFrame(y = train.consommation, x = x)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

Puisqu'il sagit d'une variable explicative catégorielle nominale, et non ordinale, l'hypothèse de linéarité entre les catégories est automatiquement à rejeter

In [ ]:
plot_explanatory_variable(model, data, "Transmission")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homoscédasticité des erreurs**

La variance des erreurs semble relativement constante pour chaques transmissions observées

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

Les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

La transmission de la voiture a un pouvoir explicatif modéré sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.5 Analyse de la variable _boite_

In [ ]:
data = DataFrame(y = train.consommation, x = train.boite)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

Puisqu'il sagit d'une variable explicative catégorielle nominale, l'hypothèse de linéarité entre les catégories est automatiquement à rejeter

In [ ]:
plot_explanatory_variable(model, data, "Boite")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homoscédasticité des erreurs**

La variance des erreurs semble être plus grande pour les véhicules automatiques que les véhicules manuels

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

Les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

La boîte de la voiture a un pouvoir explicatif très faible sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.6 Analyse de la variable _annee_

In [ ]:
x = train.annee
data = DataFrame(y = train.consommation, x = x)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

Il semble avoir une relation linéaire décroissante entre l'année et la consommation d'essence d'un véhicule.

In [ ]:
plot_explanatory_variable(model, data, "Année")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homoscédasticité des erreurs**

La variance des erreurs varie d'année en année

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

Les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

L'année de la voiture a un pouvoir explicatif très faible sur la consommation d'essence.

In [ ]:
r2(model)

En résumé, on peut dire que le nombre de cylindres et la cylindrée ont un contenu explicatif significatif, le type et la transmission ont un pouvoir explicatif moderé. La boite et l'année n'ont qu'une faible influence.

Pour les variables numeriques, l'hypothese de linéarité est satisfaite.

Aucune des variables satisfait l'hypothèse de la normalité des erreurs.

La variance des erreurs n'est pas toujours la même pour toutes les erreurs.

Les erreurs sont indépendantes pour toutes les variables, ce qui est logique notre ensemble des données.

# Partie 2
## Régressions linéaires multiples

Dans cette partie nous avons fait une régression linéaire multiple avec toutes les variables explicatives et une avec les variables explicatives qui ont un pouvoir explicatif, comme vue dans la partie 1. Nous avons obtenu un rms de 0.824 sur l'ensemble de validation dans le premièr cas et un rms de 0.839 avec seulement trois variables explicatives.

**Mise en place du seed pour la séparation des données d'entraînement et de validation**

In [32]:
seed = 9235

9235

### Régression linéaire multiple utilisant toutes les variables explicatives

In [ ]:
features = [:type, :transmission, :nombre_cylindres, :cylindree, :boite, :annee]
train, valid = get_data_set(seed, features)
ols_model = linear_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(ols_model, valid))
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

### Régression linéaire multiple en utilisant que les variables explicatives ayant un pouvoir explicatif respectable

In [ ]:
features = [:type, :transmission, :cylindree]
train, valid = get_data_set(seed, features)
smaller_ols_model = linear_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(smaller_ols_model, valid))
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

### Calcul du VIF pour le modèle de régression linéaire multiple

On obtient un VIF autours de 8-9 pour la cylindrée et le nombre de cylindres, ce qui peut indiquer la présence d'une multicolinéarité problématique. Cela pourrait entraîner un surajustement aux données d'entraînement et augmenter la variance des estimateurs de coefficients de régression. La prédiction est donc plus instable et difficile.

Puisque nous n'avons que très peu de variables explicatives, nous souhaiterions éviter de passer par une analyse des composantes principales, car cela réduirait encore plus notre jeu de données. Nous essaierons plutôt de contrôler cette potentielle multicolinéarité grâce a des techniques de régularisation.

Nous nous attaquerons à ce problème dans la **Partie 3**.

In [ ]:
features = [:type, :transmission, :boite, :nombre_cylindres, :cylindree, :annee]
nominal_features = [:type, :transmission, :boite]
continuous_features = [:nombre_cylindres, :cylindree, :annee]
ordinal_features = []

train, valid = get_data_set(seed, features)
data = vcat(train, valid)
data = encode(data, nominal_features, continuous_features, ordinal_features)
features = map(x -> Symbol(x), get_updated_features(data, features))

p = length(features)
vif = Dict()
for i in 1:p
    model = linear_model(data, continuous_features, features[i])
    vif[features[i]] = 1 / (1 - r2(model))
end

filter((kv) -> kv[2] > 8, vif)

Dans les graphiques suivantes on peut voir que il y a une relation linéaire entre le nombre des cylindres et la cylindrée.

In [ ]:
# analyser la relation entre nombre des cylindres et cylindrée
set_default_plot_size(30cm, 10cm)
pp = [plot(train, x=:nombre_cylindres, y=:cylindree, Geom.point),
      plot(train, x=:cylindree, y=:nombre_cylindres, Geom.point)]
gridstack(reshape(pp, (1,2)))

# reset plotsize
set_default_plot_size(15cm, 10cm)

# Partie 3
## Régression linéaire multiple avec technique de régularisation

### Régression ridge avec validation croisée pour choix du lambda optimal

In [ ]:
features = [:type, :transmission, :boite, :nombre_cylindres, :cylindree, :annee]
nominal_features = [:type, :transmission, :boite]
continuous_features = [:nombre_cylindres, :cylindree, :annee]
ordinal_features = []

train, valid = get_data_set(seed, features)
train, valid = align_categorical_features(train, valid, features)

y_train = train[!, :consommation]
X_train = train[!, features]
X_train = encode(X_train, nominal_features, continuous_features, ordinal_features)

X_valid = valid[!, features]
X_valid = encode(X_valid, nominal_features, continuous_features, ordinal_features)

ridge_machine, λ̂ = ridge_regression_cv(X_train, y_train)

ŷ = MLJ.predict(ridge_machine, X_valid)
y_valid = valid[:, :consommation]

println("RMSE: $(rms(ŷ, y_valid))\nλ optimal: $λ̂")

On obtient, en apparence, un résultat de RMSE très comparable a celui de la régression linéaire multiple avec toutes les paramètres. On peut aussi observer que le coefficient le plus important du modèle de régression linéaire multiple, à savoir, la cylindrée, a été pénalisé dans la régression ridge. Cependant, le lambda optimal obtenu est bien petit, donc la régression ridge se comporte davantage comme une régression linéaire.

Tout cela suggère que la multicolinéarité dans les données n'est pas suffisamment élevée pour justifier l'utilisation d'une régularisation importante.

Après avoir écarté l'hypothèse de multicolinéarité, nous nous tournerons vers une approche de régression bayésienne pour tenter d'améliorer les prédiction du modèle de régression linéaire multiple dans la **Partie 4**.

In [ ]:
fitted_params(ridge_machine), ols_model

# Partie 4
## Régression bayésienne

Puisque nous encoderons nos variables explicatives nominales avec l'encodage one hot, il sera difficile de trouver une loi à priori informative pour celle-ci. 

Cependant, comme l'a montré notre analyse préliminaire, la cylindrée et le nombre de cylindres sont deux variables faciles à poser sur une échelle continue et qui ont un bon pouvoir explicatif sur la consommation d'essence. Nous nous concentrerons donc à trouver des lois à priori pour ces deux variables.

### Traitement de données antérieures

In [ ]:
# source https://www.kaggle.com/datasets/eimadevyni/car-model-variants-and-images-dataset
data = CSV.read("cars_dataset.csv", DataFrame, decimal=',')
data = filter(x -> !ismissing(x.cylinders) && !ismissing(x.engine_specs_title) && x.from_year >= 2010 && x.to_year <= 2024, data)
data = filter(x -> begin
    m = match(r"\d(\.\d)?(?=L)", x.engine_specs_title)

    return m != nothing
end, data)
data = filter(x -> !ismissing(x.combined) && x.combined != nothing, data)

# extraction de la cylindrée
data.cylindree = map(x -> begin 
    m = match(r"\d(\.\d)?(?=L)", x)

    if m != nothing
        return parse(Float64, m.match)
    end
end, data.engine_specs_title)

# extraction du nombre de cylindres
data.nombre_cylindres = map(x -> begin 
    m = match(r"\d\d?", x)

    if m != nothing
        return parse(Float64, m.match)
    end
end, data.cylinders)

# extraction de la consommation d'essence en L/100km
data.consommation = map(x -> begin
    m = match(r"\d(\.\d)?\s*?(?=L\/100\s*Km)", x)

    if m != nothing
        return parse(Float64, m.match)
    end
end, data.combined)

data.annee = round.((data.from_year .+ data.to_year) ./ 2; digits=0)
data.annee = standardize_vec(data.annee)

### Loi a priori pour la cylindrée

Nous utiliserons la loi Gamma pour modéliser la loi a priori car elle s'ajuste bien aux données de cylindrées assez assymétrique

In [ ]:
dist_cylindree = fit(Gamma, data.cylindree)

Gadfly.plot(
    layer(x->pdf(dist_cylindree, x), 0, 12, Theme(default_color=colorant"red")),
    layer(x=data.cylindree, Geom.histogram(bincount=30, density=true)),
)

### Loi a priori pour le nombre de cylindres

Nous utiliserons la loi Gamma pour modéliser la loi a priori car elle s'ajuste bien aux données de nombre de cylindres aussi assez assymétrique

In [ ]:
dist_nombre_cylindres = fit(Gamma, data.nombre_cylindres)

Gadfly.plot(
    layer(x->pdf(dist_nombre_cylindres, x), 0, 18, Theme(default_color=colorant"red")),
    layer(x=data.nombre_cylindres, Geom.histogram(bincount=30, density=true)),
)

In [ ]:
### Choix de la loi a priori pour l'année

Nous avons décidé de centré réduire l'année afin de limiter l'impact de grand nombre sur le modèle et mieux exposer la linéarité entre l'année et la consommation d'essence. Nous modéliserons la distribution de celle-ci à l'aide d'une loi Normale.

In [ ]:
dist_annee = fit(Normal, data.annee)

Gadfly.plot(
    layer(x->pdf(dist_annee, x), -4, 4, Theme(default_color=colorant"red")),
    layer(x=data.annee, Geom.histogram(bincount=30, density=true)),
)

### Signifiance de la variance de la consommation pour la loi a priori de la variance

Puisque la variance est relativement grande (~7), nous modéliserons celle-ci à l'aide d'une loi InverseGamma de paramètres alpha = 1 et beta = 2 pour avoir des ailes assez lourdes pour supporter de plus grandes variances

In [ ]:
var(data.consommation)

### Modèle de régression bayésienne

In [ ]:
features = [:type, :transmission, :boite, :nombre_cylindres, :cylindree, :annee]
nominal_features = [:type, :transmission, :boite]
continuous_features = [:nombre_cylindres, :cylindree, :annee]
ordinal_features = []

train, valid = get_data_set(seed, features)
train, valid = align_categorical_features(train, valid, features)

train.annee = standardize_vec(train.annee)
valid.annee = standardize_vec(valid.annee)

y_train = train[!, :consommation]
X_train = train[!, features]
# encodage one hot pour les variables nominales et continue pour les variables continues
X_train = encode(X_train, nominal_features, continuous_features, ordinal_features)

y_valid = valid[!, :consommation]
X_valid = valid[!, features]
X_valid = encode(X_valid, nominal_features, continuous_features, ordinal_features)

model = bayesian_regression(
    X_train, 
    y_train, 
    Dict(
        "cylindree" => dist_cylindree,
        "nombre_cylindres" => dist_nombre_cylindres,
        "annee" => dist_annee,
        "σ²" => InverseGamma(1, 2),
    )
)
# nous utiliserons l'algorithme d'échantillonage de Gibbs couplé à celui No U-Turn Sampling avec un taux d'acceptation de 65% pour 
# les lois a posteriori trop complexes pour Gibbs.
# Bien que nous avons vu Metropolis-Hastings, cet méthode d'échantillonage nous a proposé de meilleurs résultats
sampler = Gibbs(NUTS(0.65))
chain = sample(model, sampler, 1000, discard_initial = 100)

StatsPlots.plot(chain)

MethodError: MethodError: no method matching standardize(::Vector{Int64})

Closest candidates are:
  standardize(!Matched::Type{DT}, !Matched::AbstractVecOrMat{<:Real}; kwargs...) where DT<:AbstractDataTransform
   @ StatsBase ~/.julia/packages/StatsBase/XgjIN/src/transformations.jl:366


### Analyse des résultats

On obtient une mesure de RMSE légèrement meilleure par rapport au modèle de régression linéaire multiple. Cela pourrait indiquer que nos lois a priori sont bien choisies et permettent de mieux capturer la structure des données.

Toutefois, l'amélioration n'est pas très significative. Cela peut-être du aux limitations qu'impose l'encodage one-hot, ce type de représentation traite chaque catégorie comme indépendante, ignorant les relations potentielles entre elles et rend la modélisation de lois à priori informatives pour celles-ci beaucoup plus difficile. Une solution possible à ce problème serait d'encoder dans ordre particulier si possible (par exemple, tailles croissantes des types de véhicules) afin de les rendres ordinales, rendant le choix d'une loi a priori plus envisageable.

In [ ]:
ŷ = bayesian_prediction(chain, X_valid)
rms(ŷ, y_valid)

# Partie 5
## Réduction de la base de données

In [ ]:
println("Nombre de ligne du jeu de données d'entraînement : ", nrow(train))
println("Nombre de ligne unique du jeu de données : ", nrow(unique(train)))
println("Nombre de ligne unique du jeu de données sans l'année: ", nrow(unique(select(train, Not(:annee)))))
println("Nombre de valeur de consommation unique : ", nrow(unique(train, :consommation)))

Ces informations nous permettent de voir que le jeu de données contient beaucoup de ligne identiques, encore plus si on retire l'année.  
Cela indique que le model pourrais être biaisé par un poids plus important des données redondantes dans le modèles et que l'année semble ne pas apporter pas d'information significative.
  
De plus, on voit que les valeurs possibles de consommation sont limitées, ce qui peut indiquer que certaines combinaisons de caractéristique sont équivalente.  

In [ ]:
first(sort(unique(select(train, Not(:annee))), :consommation), 5)

Par exemple, on peut voir que les voitures des deux premières lignes sont légèrement différentes, mais ont la même consommation.   

In [ ]:
unique_data = sort(unique(train), :consommation)

unique_data[!, :model] .= string.(unique_data[!, :type], "_", unique_data[!, :nombre_cylindres], "_", unique_data[!, :cylindree], "_", unique_data[!, :transmission], "_", unique_data[!, :boite])

Gadfly.set_default_plot_size(38cm, 25cm)

plt = plot(
    layer(unique_data, x=:model, y=:consommation, Geom.point),
    layer(unique_data, x=:model, y=GLM.predict(lm(@formula(consommation ~ model), unique_data)), Geom.line, Theme(default_color="red")),
    Guide.xlabel("Cylindrée"), Guide.ylabel("Consommation"),
)

display(plt)

À partir des différentes caractérise (hors année) nous avons crée une nouvelle colone **model**, qui est une chaine de caractère qui représente les caractéristiques de la voiture.  
Puis nous avons utilisé cette nouvelle colonne pour afficher la distribution de la consommation avec la droite de régression.  
On peut voir que la consommation semble suivre une relation continue, celle ci augmente semble augmenter linéairement après les 3 premiers **model** puis s’accélère sur la fin.  

Les premiers **model** sont soit des voiture très spécifique, des valeurs aberrantes ou une erreur de classification par exemple une voiture hybride classé comme essence.  
Les derniers **model** correspondent à des grosses voiture type SUV ou des voitures de sport (gros cylindrée, gros nombre de cylindre) qui consomme beaucoup plus que les autres voitures.

De plus, on peut voir que pour une même **Model** on a parfois des consommation très différente, étant donnée que nous n'avons pas pris l'année pour crée le **model** nous allons voir si l'année peut expliquer ces différences.


In [ ]:
sort(unique_data[unique_data[:,:model] .== "VUS_petit_4_2.0_integrale_automatique", :], :annee)

Ici nous affichons les lignes trié par année pour le **model** : *VUS_petit_4_2.0_integrale_automatique*  
On voit pas de relation entre l'année et la consommation, on peut donc supposer que l'année n'apporte pas d'information significative pour prédire la consommation.

In [ ]:
sort(unique_data[unique_data[:,:model] .== "VUS_petit_4_2.4_traction_automatique", :], :annee)


Ici nous affichons les lignes trié par année pour le **model** : *VUS_petit_4_2.4_traction_automatique*  
Ici peut legèrement voir une tendance à la baisse de la consommation en fonction de l'année mais pas de manière significative.

Cela montre que mous pourrions combiner les différentes consommation pour un même modèle en prenant la moyenne ou le mode.

In [ ]:
mean_consommation = combine(groupby(unique_data, :model), :consommation => mean)
unique_data_mean = innerjoin(unique(select(unique_data, Not(:annee))), mean_consommation, on=:model)

mode_consommation = combine(groupby(unique_data, :model), :consommation => mode)
unique_data_mode = innerjoin(unique(select(unique_data, Not(:annee))), mode_consommation, on=:model);

unique_data_mean = select(unique_data_mean, Not(:consommation, :model))
unique_data_mode = select(unique_data_mode, Not(:consommation, :model))

On va alors réduire la base de données en prenant la moyenne ou le mode de la consommation pour un même **model**.  

In [ ]:
X₀_mean = coerce(select(unique_data_mean, Not(:consommation_mean)), :type=>Multiclass, :transmission=>Multiclass, :boite=>Multiclass)
X₀_mean[!, :nombre_cylindres] = Float64.(X₀_mean[!, :nombre_cylindres])

X₀_mode = coerce(select(unique_data_mode, Not(:consommation_mode)), :type=>Multiclass, :transmission=>Multiclass, :boite=>Multiclass)
X₀_mode[!, :nombre_cylindres] = Float64.(X₀_mode[!, :nombre_cylindres])

y₀_mean = unique_data_mean[!, :consommation_mean]
y₀_mode = unique_data_mode[!, :consommation_mode];

In [ ]:
function one_hot_encode(data, columns)
    for column in columns
        coerce!(data, column => Multiclass)
    end

    one_hot = machine(OneHotEncoder(), data)
    fit!(one_hot, verbosity=0)
    return MLJ.transform(one_hot, data)
end

In [ ]:
# TODO a voir si on deplace
LinearRegressor = @load LinearRegressor pkg=MLJLinearModels verbosity=0
function create_linear_machine(X, y)
    linear_machine = machine(LinearRegressor(), X, y)
    fit!(linear_machine , verbosity=0)
    return linear_machine
end

X₀_mean = one_hot_encode(X₀_mean, [:type, :transmission, :boite])
X₀_mode = one_hot_encode(X₀_mode, [:type, :transmission, :boite])

linear_machine_mean = create_linear_machine(X₀_mean, y₀_mean)
linear_machine_mode = create_linear_machine(X₀_mode, y₀_mode)

In [ ]:
evaluate!(linear_machine_mean, resampling=CV(shuffle=true), measure=rms)

In [ ]:
evaluate!(linear_machine_mean, resampling=CV(shuffle=true), measure=rms)


Les mesures de RMSE par cross validation pour notre consommation transformer par **model** ne sont pas très bonnes, ... 

In [ ]:
test_data = CSV.read("test.csv", DataFrame, decimal=',')

test_data = one_hot_encode(test_data, [:type, :transmission, :boite])

test_data = select(test_data, Not(:annee))



ŷ_mean = MLJ.predict(linear_machine_mean, test_data)
ŷ_mode = MLJ.predict(linear_machine_mode, test_data)

n = nrow(test_data)

id = 1:n

df_pred = DataFrame(id=id, consommation=ŷ_mean)
CSV.write("benchmark.csv", df_pred)

best_pred = CSV.read("current_best.csv", DataFrame, decimal='.')
current_pred = CSV.read("benchmark.csv", DataFrame, decimal='.')
println("rmse between best and current (mean): ", rms(best_pred.consommation, current_pred.consommation))

df_pred = DataFrame(id=id, consommation=ŷ_mode)
CSV.write("benchmark.csv", df_pred)

current_pred = CSV.read("benchmark.csv", DataFrame, decimal='.')
println("rmse between best and current (mode): ", rms(best_pred.consommation, current_pred.consommation))